# 02_preprocess_feature
Tiền xử lý và feature engineering


In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

df = pd.read_csv("../data/raw/weatherHistory.csv", parse_dates=["Formatted Date"])
# Đảm bảo cột Formatted Date là datetime; nếu không đọc được sẽ trở thành NaT
df["Formatted Date"] = pd.to_datetime(df["Formatted Date"], errors="coerce")
if df["Formatted Date"].isna().any():
    n_invalid = df["Formatted Date"].isna().sum()
    print(f"{n_invalid} giá trị Formatted Date không hợp lệ, sẽ drop")
    df = df.dropna(subset=["Formatted Date"]).reset_index(drop=True)
df = df.sort_values("Formatted Date")
df["Temperature (C)"] = df["Temperature (C)"].interpolate().fillna(method="bfill")
df["Humidity"] = df["Humidity"].interpolate().fillna(method="bfill")
df["year"] = df["Formatted Date"].dt.year
df["month"] = df["Formatted Date"].dt.month
df["hour"] = df["Formatted Date"].dt.hour
df["temp_lag1"] = df["Temperature (C)"].shift(1)
df["temp_lag24"] = df["Temperature (C)"].shift(24)
df = df.dropna().reset_index(drop=True)
X = df[["Temperature (C)","Humidity","year","month","hour","temp_lag1","temp_lag24"]]
cat_cols=["Summary"] if "Summary" in df.columns else []
# Với sklearn >=1.2, tham số `sparse` được thay bằng `sparse_output`
transformer=ColumnTransformer([("num",StandardScaler(),X.columns), ("cat",OneHotEncoder(sparse_output=False, handle_unknown="ignore"),cat_cols)], remainder="drop")
X_trans=transformer.fit_transform(df)
print("processed shape", X_trans.shape)
df.to_csv("../data/processed/weather_featured.csv", index=False)


39984 giá trị Formatted Date không hợp lệ, sẽ drop
processed shape (56010, 32)


C:\Users\Admin\AppData\Local\Temp\ipykernel_30524\364984447.py:14: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df["Temperature (C)"] = df["Temperature (C)"].interpolate().fillna(method="bfill")
C:\Users\Admin\AppData\Local\Temp\ipykernel_30524\364984447.py:15: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df["Humidity"] = df["Humidity"].interpolate().fillna(method="bfill")
